In [ ]:
"""k-NN (emlearn) - varredura, geracao da lib C e verificacao exata.

A lib entregue (lib/knob-senses-knn) e' escrita pela celula de geracao: re-executar o
notebook reproduz os arquivos byte a byte (mesma semente). O conferidor g++ prova que o
C do firmware reproduz o emulador bit a bit (em mW)."""

import subprocess
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / 'models' / 'src'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import emlearn

from common import contract, data, paths, verify

import knn

RAIZ = paths.ROOT

SEMENTE = contract.SEED
FEATURES = contract.RF_FEATURES
ALVO = contract.TARGET

np.random.seed(SEMENTE)

print("emlearn :", emlearn.__version__)
print("features:", ", ".join(FEATURES))
print("g++ host:", subprocess.run([verify.GXX, "--version"], capture_output=True,
                                 text=True).stdout.splitlines()[0])
print("entrega :", paths.LIB_KNN.relative_to(RAIZ))

In [9]:
treino = data.load_split("train")
valid = data.load_split("validation")
teste = data.load_split("test")

# Sanidade: o csv precisa estar em GRANDEZA FISICA. Se alguem voltar a normalizar no
# prep, estes limites quebram AQUI - e nao em campo, com o firmware normalizando duas
# vezes e a previsao colapsando num valor fixo.
for nome, d in (("train", treino), ("validation", valid), ("test", teste)):
    data.check_physical_scale(nome, d)

print(f"treino {len(treino)} | validacao {len(valid)} | teste {len(teste)}")

treino 1820 | validacao 980 | teste 354


In [10]:
# Media e desvio SO do split de treino (o mesmo contrato do RF).
MEDIA, DESVIO = data.fit_normalization(treino)

X_tre = treino[list(FEATURES)]
y_tre = treino[ALVO].to_numpy()
X_tes = teste[list(FEATURES)]
y_tes = teste[ALVO].to_numpy()

# Quantizacao int16 (imposta pelo emlearn): z -> lrintf(z*1000) clampado em +-8. O
# clamp nao e' estetica: garante soma de quadrados < 2**30, onde o eml_isqrt do
# emlearn travaria (o `q <<= 2` estoura o int32). Os rotulos viram miliwatts.
Zq_tre = knn.quantiza_z(X_tre, MEDIA, DESVIO)
yq_tre = knn.quantiza_alvo(y_tre)
Zq_tes = knn.quantiza_z(X_tes, MEDIA, DESVIO)

print("quantizado: treino", Zq_tre.shape, "| z min/max", Zq_tre.min(), Zq_tre.max())

quantizado: treino (1820, 3) | z min/max -3320 2364


In [11]:
# Tamanho x acuracia: quantos prototipos e quantos vizinhos. sklearn sobre os dados
# QUANTIZADOS (rapido); a exatidao fica com o emulador mais abaixo. RAM = scratch do
# emlearn (8 B por prototipo) - o custo de usar o pipeline dele como vem.
t0 = time.perf_counter()
tabela = knn.sweep(Zq_tre, yq_tre, Zq_tes, y_tes,
                   n_opcoes=(1000, 2000, 4000, len(treino)), k_opcoes=knn.SWEEP_K)
print(f"{time.perf_counter() - t0:.1f} s\n")
print(tabela.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

0.3 s

 prototipos  k      mae       r2  flash_kb  ram_kb
       1000  1 112.9661 -19.0515    7.8125  7.8125
       1000  3 111.8669 -18.4421    7.8125  7.8125
       1000  5 111.6840 -18.3210    7.8125  7.8125
       1000  9 111.5856 -18.2631    7.8125  7.8125
       1000 15 111.9990 -18.3669    7.8125  7.8125
       1820  1 114.2931 -19.5170   14.2188 14.2188
       1820  3 112.6757 -18.7103   14.2188 14.2188
       1820  5 112.2161 -18.5383   14.2188 14.2188
       1820  9 111.8443 -18.3656   14.2188 14.2188
       1820 15 111.7144 -18.2974   14.2188 14.2188
       1820  1 114.2931 -19.5170   14.2188 14.2188
       1820  3 112.6757 -18.7103   14.2188 14.2188
       1820  5 112.2161 -18.5383   14.2188 14.2188
       1820  9 111.8443 -18.3656   14.2188 14.2188
       1820 15 111.7144 -18.2974   14.2188 14.2188
       1820  1 114.2931 -19.5170   14.2188 14.2188
       1820  3 112.6757 -18.7103   14.2188 14.2188
       1820  5 112.2161 -18.5383   14.2188 14.2188
       1820  9 111.8443 

In [12]:
# A regra do spec: (prototipos, k) crescente; vence a primeira config a <= 5 % do
# melhor MAE - o menor conjunto que nao custa acuracia de verdade.
K_ESCOLHIDO, N_ESCOLHIDO, PORQUE = knn.escolhe_config(tabela)
print(f"escolha: k = {K_ESCOLHIDO}, prototipos = {N_ESCOLHIDO}")
print(PORQUE)

escolha: k = 1, prototipos = 1000
MAE 112.9661 kW (melhor da varredura: 111.5856; limite de 5 %: 117.1649) com o menor conjunto/k que atende


In [13]:
linha = tabela[(tabela["k"] == K_ESCOLHIDO) & (tabela["prototipos"] == N_ESCOLHIDO)].iloc[0]
escritos = knn.generate_c_lib(X_tre, y_tre, MEDIA, DESVIO,
                              mae=linha["mae"], r2=linha["r2"],
                              k=K_ESCOLHIDO, n=N_ESCOLHIDO)
for caminho in sorted(escritos):
    print(f"  {caminho.relative_to(RAIZ)!s:52} {escritos[caminho]:,} bytes")
print("\nOK: nenhum marcador sobrou nos arquivos gerados")

  lib\knob-senses-knn\library.json                     516 bytes
  lib\knob-senses-knn\normalizacao.json                995 bytes
  lib\knob-senses-knn\README.md                        1,678 bytes
  lib\knob-senses-knn\src\knob_senses_knn.cpp          3,315 bytes
  lib\knob-senses-knn\src\knob_senses_knn.h            2,304 bytes

OK: nenhum marcador sobrou nos arquivos gerados


In [14]:
conferidor = verify.write_checker(dest=paths.SCRATCH / "knn_checker.cpp",
                                     header="knob_senses_knn.h", func="knn_pca_prever",
                                     n_macro="KNOB_KNN_N_FEATURES",
                                     n_features=len(FEATURES))
exe = verify.compile_host(conferidor, paths.SCRATCH / "knn_checker.exe",
                             lib_src=paths.LIB_KNN_SRC, impl="knob_senses_knn.cpp")
print("g++: compilou o conferidor junto com o knob_senses_knn.cpp da lib")

# As linhas de teste em BRUTO pelo C...
pred_c = np.asarray(verify.predict(exe, X_tes.to_numpy()), dtype=np.float64)
pred_c_mw = np.rint(pred_c * 1000).astype(np.int64)

# ...e o emulador replicando o MESMO algoritmo (float32 da consulta incluso):
idx = knn._prototipos(len(treino), N_ESCOLHIDO)
emu = knn.preve_emulado_lote(knn.quantiza_z(X_tre.iloc[idx], MEDIA, DESVIO),
                             knn.quantiza_alvo(y_tre[idx]),
                             knn.quantiza_z_c(X_tes, MEDIA, DESVIO), K_ESCOLHIDO)
assert np.array_equal(pred_c_mw, emu), "o C divergiu do emulador - bug de verdade"
print(f"OK: {len(pred_c)} linhas identicas ao emulador (bit a bit, em mW)")

# Referencia arbitraria: sklearn nos mesmos ints. As diferencas sao EMPATES do
# floor-sqrt do emlearn (a distancia inteira empata mais que o float do sklearn).
from sklearn.neighbors import KNeighborsRegressor
sk = KNeighborsRegressor(n_neighbors=K_ESCOLHIDO, algorithm="brute", metric="euclidean",
                         weights="uniform").fit(
    knn.quantiza_z(X_tre.iloc[idx], MEDIA, DESVIO).astype(np.float64),
    knn.quantiza_alvo(y_tre[idx]).astype(np.float64))
dif = np.abs(pred_c - sk.predict(knn.quantiza_z(X_tes, MEDIA, DESVIO).astype(np.float64)) / 1000)
print(f"vs sklearn: {int((dif * 1000 > 0.75).sum())} de {len(dif)} linhas > 0,75 mW "
      f"(maior {dif.max():.4f} kW) - empates de fronteira, nao erro")

g++: compilou o conferidor junto com o knob_senses_knn.cpp da lib


AssertionError: o C divergiu do emulador - bug de verdade

In [ ]:
erro = np.abs(pred_c - y_tes)
amplitude = float(pred_c.max() - pred_c.min())
print(f"o caminho inteiro, nas {len(X_tes)} linhas de test.csv:")
print(f"  MAE        : {erro.mean():.4f} kW")
print(f"  amplitude  : {amplitude:.4f} kW")
print(f"  primeira   : real {y_tes[0]:.4f}  previsto {pred_c[0]:.4f}")

# Duas guardas: contrato quebrado (normalizacao/quantizacao dupla) e previsao constante.
assert erro.mean() < 0.35, f"MAE {erro.mean():.4f} kW pelo caminho da rede - o contrato quebrou"
assert amplitude > 0.1, "amplitude baixa: previsao praticamente constante"
print("\nOK: o caminho csv -> quantiza -> lib devolve previsao util")

In [ ]:
medida = verify.measure_xtensa(lib_src=paths.LIB_KNN_SRC,
                                  impl="knob_senses_knn.cpp",
                                  obj_file=paths.SCRATCH / "knob_senses_knn.o")
flash, ram = medida["flash"], medida["ram"]
print(f"  FLASH  {flash:7} bytes  ({flash / 1024:.1f} KB)")
print(f"  RAM    {ram:7} bytes  ({ram / 1024:.1f} KB)  = 8 B x {N_ESCOLHIDO} prototipos + bss")
print(f"  para comparar: o RF ocupa ~68 KB de flash e 0 de RAM")

In [ ]:
print("k-NN (emlearn) - procedencia")
print(f"  k = {K_ESCOLHIDO} | prototipos = {N_ESCOLHIDO} | semente = {SEMENTE}")
print(f"  MAE {erro.mean():.4f} kW | R2 {linha['r2']:.4f} no split de teste")
print(f"  flash ~{flash / 1024:.1f} KB | RAM ~{ram / 1024:.1f} KB")
print("  entrega:", paths.LIB_KNN.relative_to(RAIZ))